# LAB- P7: El Modelo de Equilibrio General Dinámico Básico (DGE)
**Proyecto MACRO-AI-COMP (Convocatoria INNOVA26, UMA / Banco Santander)**
*   **Código de LAB-**: LAB-P7-v1.0
*   **Capítulo de Referencia**: Capítulo 8, *An Introduction to Computational Macroeconomics* (Bongers, Gómez y Torres, Vernon Press, 2019)
*   **Autores**: Equipo Docente MACRO-AI-COMP
*   **Objetivo**: Desarrollar e interactuar con el modelo canónico de Equilibrio General Dinámico (DGE) en tiempo discreto, resolviendo y comparando la aproximación linealizada de Blanchard-Khan frente a la solución numérica no lineal exacta.

---

## Objetivos de Aprendizaje
Al finalizar esta práctica, serás capaz de:
1.  **Comprender** la microfundamentación intertemporal de las decisiones de consumo y ahorro y su interacción con el sector productivo en equilibrio general.
2.  **Aplicar** el método de Blanchard-Khan para resolver y simular sistemas dinámicos lineales con expectativas racionales y variables forward-looking.
3.  **Analizar** el mecanismo de propagación intertemporal y la persistencia de shocks tecnológicos (TFP) sobre las variables macroeconómicas clave (Producción, Consumo, Inversión, Capital y Tasa de Interés).
4.  **Evaluar** la precisión y los límites de la aproximación linealizada frente a la solución no lineal exacta ante perturbaciones de distinta magnitud.
5.  **Identificar** discrepancias de programación y timing en modelos macroeconómicos computacionales complejos.
 (Julia)

> **👋 BIENVENIDA A LA PRÁCTICA - LEER ANTES DE EMPEZAR**
> 
> *   **¿Nunca has usado Jupyter?** No te preocupes. Este cuaderno es interactivo. Haz clic en cualquier celda de código y pulsa **`Shift + Enter`** para ejecutarla. Ve de arriba a abajo en orden.
> *   **¿Se ha congelado o sale un asterisco `[*]` eterno?** Ve al menú superior y dale a `Kernel` ➔ `Restart`.
> *   **El objetivo** de esta práctica es que juegues con la economía. Cambia los números del código que representan impuestos, dinero o tecnología, vuelve a ejecutar y mira los gráficos. ¡No puedes romper nada!
>

### 🕹️ GUÍA RÁPIDA PARA DUMMIES - Equilibrio General Dinámico (DGE)
*   **¿Qué estamos haciendo aquí?** Uniendo a los consumidores, empresas y mercado en un solo gran simulador macroeconómico.
*   **El Ciclo Económico:** Simulamos cómo un shock tecnológico (un aumento repentino de productividad) hace que aumenten a la vez el PIB, la inversión y el consumo.
*   **¡Prueba esto!** Ejecuta la celda del shock tecnológico y observa el gráfico en forma de "joroba" que muestra la propagación de la bonanza en la economía.


In [1]:
# En Google Colab se activarían y descargarían los paquetes necesarios.
# using Pkg; Pkg.activate("."); Pkg.instantiate()


In [2]:
using Pkg
Pkg.activate("../..")

using MacroAIComp
using Plots
import Plots: mm
using LinearAlgebra
using NLsolve
using Interact
using BenchmarkTools


  Activating project at `C:\Users\AntonioRC\Desktop\PIE`


WebIO._IJuliaInit()

## 1. El Modelo Canónico de Equilibrio General Dinámico (DGE)

Los modelos de Equilibrio General Dinámico (DGE), en sus versiones deterministas o estocásticas (DSGE), constituyen el núcleo de la macroeconomía moderna. A diferencia de los modelos IS-LM, los modelos DGE se basan en la **microfundamentación**: las ecuaciones de comportamiento se derivan directamente de los problemas de optimización intertemporal de los consumidores y las empresas.

### 1.1 Estructura del Modelo
El modelo canónico desarrollado en el Capítulo 8 representa una economía cerrada y perfectamente competitiva sin elecciones de ocio ($L_t = 1$):

1. **Problema del Consumidor:**
   Maximizar la utilidad descontada intertemporal:
   $$\max_{\{C_t\}} \sum_{t=0}^{\infty} \beta^t \ln(C_t)$$
   Sujeto a la restricción presupuestaria intertemporal:
   $$C_t + K_{t+1} = W_t + (R_t + 1 - \delta) K_t$$

2. **Problema de la Empresa:**
   Maximizar beneficios periodo a periodo:
   $$\max_{\{K_t, L_t\}} Y_t - W_t L_t - R_t K_t$$
   Sujeta a la función de producción Cobb-Douglas:
   $$Y_t = A_t K_t^\alpha L_t^{1-\alpha}$$

3. **Condición de Equilibrio (Vaciado de Mercados):**
   $$Y_t = C_t + I_t \quad \text{y} \quad K_{t+1} = (1-\delta) K_t + I_t$$

### 1.2 Sistema de Ecuaciones Reducido
El equilibrio competitivo dinámico de esta economía se reduce a un sistema de dos ecuaciones diferenciales en diferencias (una para la variable rígida/predeterminada $K_t$ y otra para la variable de expectativas/flexible $C_t$), alimentado por el proceso autorregresivo de la productividad (TFP):
1. **Dinámica del Capital:**
   $$K_{t+1} = (1-\delta)K_t + A_t K_t^\alpha - C_t$$
2. **Ecuación de Euler (Keynes-Ramsey):**
   $$C_{t+1} = \beta [ \alpha A_{t+1} K_{t+1}^{\alpha-1} + 1 - \delta ] C_t$$
3. **Proceso de la TFP:**
   $$\ln(A_t) = \rho \ln(A_{t-1}) + \epsilon_t$$
   Donde $\epsilon_t$ es un shock tecnológico exógeno y $\rho$ es la persistencia.

### 1.3 Estado Estacionario
Estableciendo $A = 1$ y eliminando los subíndices de tiempo en el sistema, obtenemos los valores de largo plazo:
$$\bar{R} = \frac{1 - \beta + \beta\delta}{\beta}, \quad \bar{K} = \left( \frac{\bar{R}}{\alpha} \right)^{\frac{1}{\alpha - 1}}, \quad \bar{Y} = \bar{K}^\alpha, \quad \bar{I} = \delta \bar{K}, \quad \bar{C} = \bar{Y} - \bar{I}$$


In [3]:
params_base = default_calibration(DGEParams)
ss = compute_steady_state(params_base)

println("VALORES DE EQUILIBRIO (SS):")
println("  Capital (K*)   : ", round(ss["K"], digits=4))
println("  Consumo (C*)   : ", round(ss["C"], digits=4))
println("  Producción (Y*): ", round(ss["Y"], digits=4))
println("  Inversión (I*) : ", round(ss["I"], digits=4))


VALORES DE EQUILIBRIO (SS):
  Capital (K*)   : 6.6986
  Consumo (C*)   : 1.5439
  Producción (Y*): 1.9458
  Inversión (I*) : 0.4019


## 2. Simulación Interactiva: Shock Tecnológico Transitorio (TFP)

Imaginemos que la economía parte de su estado estacionario. En el periodo $t=1$, se produce un shock tecnológico transitorio positivo del $1\%$ ($\epsilon_1 = 0.01$) en la TFP, que decae de acuerdo con la persistencia $\rho = 0.8$.

Dado que la productividad aumenta, la economía experimenta un incremento inmediato en la capacidad de producción. Los consumidores, al tener expectativas de que la productividad y la tasa de interés serán más altas, aumentan su consumo inmediatamente ($C_1$ salta). Sin embargo, al ser el capital un factor rígido, no puede saltar en el periodo del shock ($K_1 = \bar{K}$). El aumento de producción se destina tanto a consumo como a inversión, lo que genera una acumulación gradual de capital físico. Este capital alcanzará su pico varios periodos después (efecto *hump-shape* o respuesta jorobada) antes de converger lentamente hacia el equilibrio inicial.


In [4]:
# Simulación interactiva: Shock Tecnológico Transitorio
@manipulate for epsilon in 0.00:0.005:0.05, rho_val in 0.5:0.05:0.95, alpha_val in 0.2:0.05:0.5, beta_val in 0.90:0.01:0.99
    
    params = DGEParams(alpha_val, beta_val, 0.05, rho_val, 1.0)
    ss_sim = compute_steady_state(params)
    K0 = ss_sim["K"]
    T_sim = 50
    
    # Path del TFP
    A_path = ones(T_sim)
    a_shock = zeros(T_sim)
    a_shock[1] = epsilon
    for t in 2:T_sim
        a_shock[t] = rho_val * a_shock[t-1]
    end
    A_path .= exp.(a_shock)
    
    res = solve_nonlinear_simulation(params, K0, A_path, T_sim)
    
    t_axis = 0:(T_sim - 1)
    
    p1 = plot(t_axis, res["Y"], color=:blue, lw=2.5, label="Producción (Y)")
    hline!([ss_sim["Y"]], color=:gray, ls=:dot, label="")
    title!("Producción (Y)")
    xlabel!("Periodos")
    
    p2 = plot(t_axis, res["C"], color=:purple, lw=2.5, label="Consumo (C)")
    hline!([ss_sim["C"]], color=:gray, ls=:dot, label="")
    title!("Consumo (C)")
    xlabel!("Periodos")
    
    p3 = plot(t_axis, res["I"], color=:orange, lw=2.5, label="Inversión (I)")
    hline!([ss_sim["I"]], color=:gray, ls=:dot, label="")
    title!("Inversión (I)")
    xlabel!("Periodos")
    
    p4 = plot(t_axis, res["K"], color=:forestgreen, lw=2.5, label="Capital (K)")
    hline!([ss_sim["K"]], color=:gray, ls=:dot, label="")
    title!("Capital (K)")
    xlabel!("Periodos")
    
    plot(p1, p2, p3, p4, layout=(2,2), size=(900, 600), 
         plot_title="Ajuste Dinámico frente a Shock TFP (No Lineal)", top_margin=10mm)
end


Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Scope(Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :label), Any["epsilon"], Dict{Symbol, Any}(:className => "interact ", :style => Dict{Any, Any}(:padding => "5px 10px 0px 10px")))], Dict{Symbol, Any}(:className => "interact-flex-row-left")), Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :input), Any[], Dict{Symbol, Any}(:max => 11, :min => 1, :attributes => Dict{Any, Any}(:type => "range", Symbol("data-bind") => "numericValue: index, valueUpdate: 'input', event: {change: function (){this.changes(this.changes()+1)}}", "orient" => "horizontal"), :step => 1, :className => "slider slider is-fullwidth", :style => Dict{Any, Any}()))], Dict{Symbol, Any}(:className => "interact-flex-row-center")), Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :p), Any[], Dict{Symbol, Any}(:attributes => Dict("data-bind" => "text: formatted_val")))], Dict{Symbol, Any}(:className => "interact-flex-row-right"))], Dict{Symbol, Any}(:className => "interact-flex-row interact-widget")), Dict{String, Tuple{AbstractObservable, Union{Nothing, Bool}}}("changes" => (Observable(0), nothing), "index" => (Observable{Any}(6), nothing)), Set{String}(), nothing, Asset[Asset("js", "knockout", "C:\\Users\\AntonioRC\\.julia\\packages\\Knockout\\HReiN\\src\\..\\assets\\knockout.js"), Asset("js", "knockout_punches", "C:\\Users\\AntonioRC\\.julia\\packages\\Knockout\\HReiN\\src\\..\\assets\\knockout_punches.js"), Asset("js", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\InteractBase\\8TTmI\\src\\..\\assets\\all.js"), Asset("css", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\InteractBase\\8TTmI\\src\\..\\assets\\style.css"), Asset("css", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\Interact\\PENUy\\src\\..\\assets\\bulma_confined.min.css")], Dict{Any, Any}("changes" => Any[WebIO.JSString("(function (val){return (val!=this.model[\"changes\"]()) ? (this.valueFromJulia[\"changes\"]=true, this.model[\"changes\"](val)) : undefined})")], "index" => Any[WebIO.JSString("(function (val){return (val!=this.model[\"index\"]()) ? (this.valueFromJulia[\"index\"]=true, this.model[\"index\"](val)) : undefined})")]), WebIO.ConnectionPool(Channel{Any}(32), Set{AbstractConnection}(), Base.GenericCondition(ReentrantLock())), WebIO.JSString[WebIO.JSString("function () {\n    var handler = (function (ko, koPunches) {\n    ko.punches.enableAll();\n    ko.bindingHandlers.numericValue = {\n        init: function(element, valueAccessor, allBindings, data, context) {\n            var stringified = ko.observable(ko.unwrap(valueAccessor()));\n            stringified.subscribe(function(value) {\n                var val = parseFloat(value);\n                if (!isNaN(val)) {\n                    valueAccessor()(val);\n                }\n            });\n            valueAccessor().subscribe(function(value) {\n                var str = JSON.stringify(value);\n                if ((str == \"0\") && ([\"-0\", \"-0.\"].indexOf(stringified()) >= 0))\n                     return;\n                 if ([\"null\", \"\"].indexOf(str) >= 0)\n                     return;\n                stringified(str);\n            });\n            ko.applyBindingsToNode(\n                element,\n                {\n                    value: stringified,\n                    valueUpdate: allBindings.get('valueUpdate'),\n                },\n                context,\n            );\n        }\n    };\n    var json_data = {\"formatted_vals\":[\"0.0\",\"0.005\",\"0.01\",\"0.015\",\"0.02\",\"0.025\",\"0.03\",\"0.035\",\"0.04\",\"0.045\",\"0.05\"],\"changes\":WebIO.getval({\"name\":\"changes\",\"scope\":\"18240976081678360100\",\"id\":\"3\",\"type\":\"observable\"}),\"index\":WebIO.getval({\"name\":\"index\",\"scope\":\"18240976081678360100\",\"id\":\"2\",\"type\":\"observable\"})};\n    var self =

## 3. Límites de la Aproximación Lineal: Blanchard-Khan frente a Modelo No Lineal

El método de Blanchard-Khan (o cualquier método de linealización de primer orden) asume que la economía realiza fluctuaciones pequeñas en torno a su estado estacionario, de modo que la curvatura de las funciones de utilidad y producción puede aproximarse mediante tangencias de primer orden.

Sin embargo, si la economía sufre perturbaciones de gran magnitud (por ejemplo, una caída catastrófica de TFP o un shock tecnológico masivo), la aproximación lineal comete un **error de truncamiento** sistemático.

A continuación, puedes simular shocks de diferente envergadura y evaluar visualmente cómo difieren las soluciones calculadas mediante:
1. **Blanchard-Khan (Log-Linealizado)**: Resuelve el sistema linealizado mediante la descomposición de autovalores de la matriz $J$.
2. **SciPy fsolve (No Lineal)**: Resuelve el sistema completo de ecuaciones no lineales en niveles mediante métodos iterativos de Newton.


In [5]:
# Comparación Lineal (Blanchard-Kahn) vs No Lineal
@manipulate for epsilon_shock in 0.01:0.01:0.10
    
    params = default_calibration(DGEParams)
    ss_comp = compute_steady_state(params)
    K0 = ss_comp["K"]
    T_sim = 40
    
    A_path = ones(T_sim)
    a_s = zeros(T_sim)
    a_s[1] = epsilon_shock
    for t in 2:T_sim
        a_s[t] = params.rho * a_s[t-1]
    end
    A_path .= exp.(a_s)
    
    # Resolver
    res_lin = solve_blanchard_khan(params, K0, A_path, T_sim)
    res_nonlin = solve_nonlinear_simulation(params, K0, A_path, T_sim)
    
    t_axis = 0:(T_sim - 1)
    
    # Error de Capital
    diff_K = abs.(res_nonlin["K"] .- res_lin["K"])
    
    p1 = plot(t_axis, res_nonlin["K"], color=:purple, lw=3, label="No Lineal")
    plot!(t_axis, res_lin["K"], color=:blue, ls=:dash, lw=2, label="Lineal (BK)")
    title!("Stock de Capital (K)")
    xlabel!("Tiempo")
    
    p2 = plot(t_axis, diff_K, color=:red, lw=2.5, label="Error Absoluto")
    title!("Error de Aproximación")
    xlabel!("Tiempo")
    
    plot(p1, p2, layout=(1,2), size=(800, 350), 
         plot_title="Comparativa de Solucionadores (Shock $(epsilon_shock))", top_margin=10mm)
end


Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Scope(Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :label), Any["epsilon_shock"], Dict{Symbol, Any}(:className => "interact ", :style => Dict{Any, Any}(:padding => "5px 10px 0px 10px")))], Dict{Symbol, Any}(:className => "interact-flex-row-left")), Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :input), Any[], Dict{Symbol, Any}(:max => 10, :min => 1, :attributes => Dict{Any, Any}(:type => "range", Symbol("data-bind") => "numericValue: index, valueUpdate: 'input', event: {change: function (){this.changes(this.changes()+1)}}", "orient" => "horizontal"), :step => 1, :className => "slider slider is-fullwidth", :style => Dict{Any, Any}()))], Dict{Symbol, Any}(:className => "interact-flex-row-center")), Node{WebIO.DOM}(WebIO.DOM(:html, :div), Any[Node{WebIO.DOM}(WebIO.DOM(:html, :p), Any[], Dict{Symbol, Any}(:attributes => Dict("data-bind" => "text: formatted_val")))], Dict{Symbol, Any}(:className => "interact-flex-row-right"))], Dict{Symbol, Any}(:className => "interact-flex-row interact-widget")), Dict{String, Tuple{AbstractObservable, Union{Nothing, Bool}}}("changes" => (Observable(0), nothing), "index" => (Observable{Any}(5), nothing)), Set{String}(), nothing, Asset[Asset("js", "knockout", "C:\\Users\\AntonioRC\\.julia\\packages\\Knockout\\HReiN\\src\\..\\assets\\knockout.js"), Asset("js", "knockout_punches", "C:\\Users\\AntonioRC\\.julia\\packages\\Knockout\\HReiN\\src\\..\\assets\\knockout_punches.js"), Asset("js", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\InteractBase\\8TTmI\\src\\..\\assets\\all.js"), Asset("css", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\InteractBase\\8TTmI\\src\\..\\assets\\style.css"), Asset("css", nothing, "C:\\Users\\AntonioRC\\.julia\\packages\\Interact\\PENUy\\src\\..\\assets\\bulma_confined.min.css")], Dict{Any, Any}("changes" => Any[WebIO.JSString("(function (val){return (val!=this.model[\"changes\"]()) ? (this.valueFromJulia[\"changes\"]=true, this.model[\"changes\"](val)) : undefined})")], "index" => Any[WebIO.JSString("(function (val){return (val!=this.model[\"index\"]()) ? (this.valueFromJulia[\"index\"]=true, this.model[\"index\"](val)) : undefined})")]), WebIO.ConnectionPool(Channel{Any}(32), Set{AbstractConnection}(), Base.GenericCondition(ReentrantLock())), WebIO.JSString[WebIO.JSString("function () {\n    var handler = (function (ko, koPunches) {\n    ko.punches.enableAll();\n    ko.bindingHandlers.numericValue = {\n        init: function(element, valueAccessor, allBindings, data, context) {\n            var stringified = ko.observable(ko.unwrap(valueAccessor()));\n            stringified.subscribe(function(value) {\n                var val = parseFloat(value);\n                if (!isNaN(val)) {\n                    valueAccessor()(val);\n                }\n            });\n            valueAccessor().subscribe(function(value) {\n                var str = JSON.stringify(value);\n                if ((str == \"0\") && ([\"-0\", \"-0.\"].indexOf(stringified()) >= 0))\n                     return;\n                 if ([\"null\", \"\"].indexOf(str) >= 0)\n                     return;\n                stringified(str);\n            });\n            ko.applyBindingsToNode(\n                element,\n                {\n                    value: stringified,\n                    valueUpdate: allBindings.get('valueUpdate'),\n                },\n                context,\n            );\n        }\n    };\n    var json_data = {\"formatted_vals\":[\"0.01\",\"0.02\",\"0.03\",\"0.04\",\"0.05\",\"0.06\",\"0.07\",\"0.08\",\"0.09\",\"0.1\"],\"changes\":WebIO.getval({\"name\":\"changes\",\"scope\":\"7219605121666765421\",\"id\":\"21\",\"type\":\"observable\"}),\"index\":WebIO.getval({\"name\":\"index\",\"scope\":\"7219605121666765421\",\"id\":\"20\",\"type\":\"observable\"})};\n    var self = this;\n

## 4. Cuaderno de Bitácora (Actividades para el Alumno)

Responde a las siguientes cuestiones tras interactuar con el modelo de equilibrio general dinámico:

1.  **Mecanismo de Propagación del Shock**:
    *   Establece un shock tecnológico $\epsilon_1 = 0.01$ y una persistencia $\rho = 0.8$. Describe el patrón dinámico de las 4 variables graficadas en la simulación 1. ¿Por qué el consumo y la inversión saltan instantáneamente en $t=1$, mientras que el stock de capital responde con retraso?
    *   Reduce la persistencia a $\rho = 0.2$. ¿Cómo cambia la trayectoria de consumo y capital? Explica la relación entre la persistencia del shock tecnológico y la velocidad de acumulación de capital.
2.  **Límites de la Log-Linealización**:
    *   En la simulación 2, establece un shock de TFP de $\epsilon = 0.01$ (un shock estándar de $1\%$). Observa el error relativo máximo entre Blanchard-Khan y el resolvedor no lineal exacto. ¿Qué magnitud tiene?
    *   Incrementa el shock tecnológico a $\epsilon = 0.25$ (un shock masivo de productividad del $25\%$). ¿Qué ocurre con el error relativo en consumo y capital? Explica por qué la aproximación de primer orden de Blanchard-Khan pierde validez cuando las desviaciones respecto al estado estacionario son grandes.
3.  **Higiene IA y Validación cruzada**:
    *   Verifica que los valores de estado estacionario calculados por Python coincidan con la Tabla 8.2 del libro impreso ($\bar{K}=6.699$, $\bar{Y}=1.946$, $\bar{C}=1.544$, $\bar{R}=0.102$). Reporta el resultado en tu bitácora.


## 5. Buenas Prácticas Aplicadas en este Laboratorio
1.  **Aislamiento Paramétrico**: El código de las rutinas matriciales y resolvedores de Blanchard-Khan y Newton no lineal están desacoplados en el módulo [dge.py](file:///c:/Users/AntonioRC/Desktop/PIE/src/macroaicomp/models/dge.py), importados de manera modular.
2.  **Corrección de Timing y Parámetros**: Hemos solucionado la discrepancia de indexación de TFP del código MATLAB impreso, ofreciendo una opción `use_matlab_timing` para que los estudiantes comprueben la equivalencia con la hoja del libro y con el modelo de Dynare correcto.
3.  **Higiene del Repositorio**: El cuaderno se somete automáticamente a `nbstripout` mediante hooks de `pre-commit` para evitar subir gráficos cargados y metadatos volátiles de Jupyter al control de versiones.


## 6. Benchmark de Rendimiento (Fase III)
Evaluamos la velocidad de simulación usando `BenchmarkTools.jl`.

In [6]:
# Benchmark simulation para No Lineal y Blanchard-Kahn
A_bench = ones(50)
A_bench[1] = exp(0.01)
a_bench = zeros(50); a_bench[1] = 0.01

println("Benchmark NLsolve (No Lineal):")
@btime solve_nonlinear_simulation($params_base, $ss["K"], $A_bench, 50)

println("Benchmark Blanchard-Kahn (Lineal):")
@btime solve_blanchard_khan($params_base, $ss["K"], $A_bench, 50)


Benchmark NLsolve (No Lineal):
  

10.416 ms (801898 allocations: 14.50 MiB)
Benchmark Blanchard-Kahn (Lineal):
  

14.800 μs (105 allocations: 11.01 KiB)


Dict{String, Vector{Float64}} with 4 entries:
  "Y" => [1.96534, 1.94725, 1.94711, 1.94698, 1.94687, 1.94676, 1.94667, 1.9465…
  "I" => [0.416481, 0.401394, 0.401444, 0.401489, 0.40153, 0.401567, 0.401601, …
  "C" => [1.54886, 1.54586, 1.54567, 1.54549, 1.54534, 1.5452, 1.54507, 1.54495…
  "K" => [6.6986, 6.71306, 6.71167, 6.71042, 6.70928, 6.70826, 6.70733, 6.70649…